In [1]:
import qubx
from qubx import logger

%qubxd

%load_ext autoreload
%autoreload 2

from qubx.backtester.simulator import simulate
from qubx.core.basics import DataType, Signal
from qubx.core.interfaces import IStrategy, IStrategyContext, IStrategyInitializer, MarketEvent, TriggerEvent
from qubx.core.lookups import lookup
from qubx.data.registry import StorageRegistry
from qubx.data.transformers import TypedGenericSeries
from qubx.utils.time import handle_start_stop


⠀⠀⡰⡖⠒⠒⢒⢦⠀⠀   
⠀⢠⠃⠈⢆⣀⣎⣀⣱⡀  QUBX | Quantitative Backtesting Environment 
⠀⢳⠒⠒⡞⠚⡄⠀⡰⠁         (c) 2025, ver. 0.7.40.dev8
⠀⠀⠱⣜⣀⣀⣈⣦⠃⠀⠀⠀ 
        


## Datareaders 

In [3]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor.get_exchanges()

['BINANCE.UM', 'HYPERLIQUID', 'KRAKEN']

In [ ]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("AAVEUSDT", "ohlc(1h)")

In [ ]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("BTCUSDT", "quote")

In [ ]:
stor.get_reader("KRAKEN", "SWAP").get_time_range("AAVEUSD", "ohlc(1h)")

In [ ]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").to_records()[:3]

In [ ]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote", None, None).transform(TypedGenericSeries())

In [11]:
_X1 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 1000, "high": 1100, "low": 950, "close": 1050 },
    "2025-01-01 01:00": {"open": 1050, "high": 1200, "low": 900, "close": 1075 },
    "2025-01-01 02:00": {"open": 1075, "high": 1300, "low": 1050, "close": 1100 },
}, orient="index")
_X1.index = pd.DatetimeIndex(_X1.index)

_X2 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 100, "high": 110, "low": 95, "close": 105 },
    "2025-01-01 01:00": {"open": 105, "high": 120, "low": 90, "close": 107 },
    "2025-01-01 02:00": {"open": 107, "high": 130, "low": 105, "close": 110 },
}, orient="index")
_X2.index = pd.DatetimeIndex(_X2.index)

In [ ]:
stor = StorageRegistry.get("handy", data={
    "BINANCE.UM:SWAP:BTCUSDT": _X1,
    "BINANCE.UM:SWAP:ETHUSDT": _X2
})
r = stor["BINANCE.UM", "SWAP"]
r.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)")

## Smoke test run

In [59]:
stor = StorageRegistry.get("csv::../../tests/data/storages/multi/")
# stor = StorageRegistry.get("qdb::quantlab")
stor.get_exchanges()

['BINANCE.UM', 'KRAKEN.F', 'HYPERLIQUID']

In [60]:
rr = stor.get_reader("BINANCE.UM", "SWAP")
print(rr.get_data_types("BTCUSDT"))
rr.read("BTCUSDT", "features", "2026-01-01", "now").to_records()[:4]

[quote, 'ohlc(1h)', 'features']


[[2026-01-01T01:00:00.000000000]	 {'f1': 1.0, 'f2': 11.0},
 [2026-01-01T02:00:00.000000000]	 {'f1': 2.0, 'f2': 21.0},
 [2026-01-01T03:00:00.000000000]	 {'f1': 3.0, 'f2': 31.0},
 [2026-01-01T04:00:00.000000000]	 {'f1': 4.0, 'f2': 41.0}]

In [61]:
class Test1(IStrategy):
    def on_init(self, initializer: IStrategyInitializer):
        initializer.set_event_schedule("1h")

        # initializer.set_base_subscription("ohlc(1h)")
        initializer.set_base_subscription("ohlc_quotes(1h)")
        initializer.subscribe("ohlc_trades(1h)")

        # initializer.set_base_subscription("ohlc_trades(1h)")
        # initializer.subscribe("ohlc_quotes(1h)")

        # initializer.set_base_subscription("ohlc_trades(1h)")
        # initializer.subscribe("features")
        # self._show = True

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent) -> list[Signal] | Signal | None:
        # qq = ''
        # for i in ctx.instruments:
        #     qq += f" | {i}: {ctx.quote(i).mid_price()}"
        # if data.type == "quote":

        # if self._show and data.type == "quote":
        #     self._show = False
        #     logger.info(data)

        # if data.type == "features":
            # logger.info(data)
        #     logger.info(data)
        #     self._show = True
        logger.info(data)

    def on_event(self, ctx: IStrategyContext, event: TriggerEvent) -> list[Signal] | Signal | None:
        logger.info(" | ".join([f"{i}: {ctx.quote(i).mid_price()}" for i in ctx.instruments]))

In [62]:
simulate(
    Test1(), 
    data=stor, capital=1000, 
    # start="2023-06-01", stop="2023-06-02",
    start="2026-01-01 00:00", stop="2026-01-01 05:00",
    instruments=[
        "BINANCE.UM:SWAP:BTCUSDT",
        # "KRAKEN.F:SWAP:BTCUSD",
        # "HYPERLIQUID:SWAP:BTCUSDC",
    ], 
    debug="DEBUG"
);

2026-02-12 17:09:54.887 [ 🐞 ] (runner) [Simulator] :: Preparing simulated trading on ['BINANCE.UM'] for 1000 USDT
2026-02-12 17:09:54.887 [ ℹ️ ] (data) SimulatedDataProvider.BINANCE.UM is initialized
2026-02-12 17:09:54.888 [ ℹ️ ] (processing) Setting event schedule to 1h
2026-02-12 17:09:54.889 [ 🐞 ] (context) [StrategyContext] :: Auto-assigned SimulationTransferManager
2026-02-12 17:09:54.889 [ 🐞 ] (runner) [SimulationRunner] :: Setting warmups: {'ohlc_quotes(1h)': '2h'}


2026-01-01 00:00:00.000 [🐞] (runner) [SimulationRunner] :: Running simulation from 2026-01-01 00:00:00 to 2026-01-01 05:00:00


Simulating:   0%|          | 0/100 [00:00<?, ?%/s]

2026-01-01 00:00:00.000 [🐞] (data)  | Warming up ('ohlc_quotes(1h)', BINANCE.UM:SWAP:BTCUSDT) -> 2h
2026-01-01 00:00:00.000 [ℹ️] (runner) SimulationRunner ::: Simulation started at 2026-01-01 00:00:00 :::
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Invoking Test1 on_warmup_finished
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 warmup finished completed
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Invoking Test1 on_fit
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 is fitted
2026-01-01 00:00:05.000 [ℹ️] (1995596196) MarketEvent(time=2026-01-01T00:00:05.000000000, type=trade, instrument=BINANCE.UM:SWAP:BTCUSDT, data=[2026-01-01T00:00:05.000000000]	87608.30000 (59.54) ??? )
2026-01-01 00:24:00.000 [ℹ️] (1995596196) MarketEvent(time=2026-01-01T00:24:00.000000000, type=quote, instrument=BINANCE.UM:SWAP:BTCUSDT, data=[2026-01-01T00:24:00.000000000]	87600.00000 (1000000000.00) | 87600.00000 (1000000000.00))


2026-01-01 00:59:55.000 [ℹ️] (1995596196) MarketEvent(time=2026-01-01T00:59:55.000000000, type=quote, instrument=BINANCE.UM:SWAP:BTCUSDT, data=[2026-01-01T00:59:55.000000000]	87773.40000 (1000000000.00) | 87773.40000 (1000000000.00))
2026-01-01 01:00:00.000 [ℹ️] (1995596196) BINANCE.UM:SWAP:BTCUSDT: 87773.4
2026-01-01 01:00:05.000 [ℹ️] (1995596196) MarketEvent(time=2026-01-01T01:00:05.000000000, type=quote, instrument=BINANCE.UM:SWAP:BTCUSDT, data=[2026-01-01T01:00:05.000000000]	87773.40000 (1000000000.00) | 87773.40000 (1000000000.00))
2026-01-01 01:00:05.000 [ℹ️] (1995596196) MarketEvent(time=2026-01-01T01:00:05.000000000, type=trade, instrument=BINANCE.UM:SWAP:BTCUSDT, data=[2026-01-01T01:00:05.000000000]	87773.40000 (0.98) ??? )
2026-01-01 01:24:00.000 [ℹ️] (1995596196) MarketEvent(time=2026-01-01T01:24:00.000000000, type=quote, instrument=BINANCE.UM:SWAP:BTCUSDT, data=[2026-01-01T01:24:00.000000000]	87773.30000 (1000000000.00) | 87773.30000 (1000000000.00))
2026-01-01 01:24:00.000

## Update csv storage for test

In [28]:
r1 = StorageRegistry.get("qdb::quantlab")["BINANCE.UM", "SWAP"]
r2 = StorageRegistry.get("qdb::quantlab")["KRAKEN", "SWAP"]
r3 = StorageRegistry.get("qdb::quantlab")["HYPERLIQUID", "SWAP"]

In [ ]:
r1.get_time_range("BTCUSDT", "quote")

In [ ]:
r2.get_time_range("BTCUSD", "quote")

In [ ]:
r3.get_time_range("BTCUSDC", "quote")

In [ ]:
q1 = r1.read("BTCUSDT", "quote", "2026-01-01", "2026-01-03")
q2 = r2.read("BTCUSD", "quote", "2026-01-01", "2026-01-03")
q3 = r3.read("BTCUSDC", "quote", "2026-01-01", "2026-01-03")

q1.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/BINANCE.UM/SWAP/BTCUSDT.quote.csv.gz")
q2.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/KRAKEN.F/SWAP/BTCUSD.quote.csv.gz")
q3.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/HYPERLIQUID/SWAP/BTCUSDC.quote.csv.gz")

In [ ]:
# qqs

# q1 = r1.read("BTCUSDT", "quote", "2026-01-01", "2026-01-03")

In [35]:
sm = "BCH"
q1 = r1.read(f"{sm}USDT", "ohlc(1h)", "2025-01-01", "2026-01-03")
q2 = r2.read(f"{sm}USD", "ohlc(1h)", "2025-01-01", "2026-01-03")
q3 = r3.read(f"{sm}USDC", "ohlc(1h)", "2025-01-01", "2026-01-03")

q1.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/BINANCE.UM/SWAP/{sm}USDT.1H.csv.gz")
q2.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/KRAKEN.F/SWAP/{sm}USD.1H.csv.gz")
q3.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/HYPERLIQUID/SWAP/{sm}USDC.1H.csv.gz")